In [1]:
!pip install -q transformers datasets evaluate jiwer accelerate
!pip install -q yt-dlp pydub ffmpeg-python kagglehub
!apt-get install -q -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.1 MB/s eta 0:00:00a 0:00:01
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.


In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [3]:
dataset_path = "/kaggle/input/datasets/mayarjao/arabic-tts/arabic_tts"
meta_path    = os.path.join(dataset_path, "metadata-wav.csv")
wavs_dir     = os.path.join(dataset_path, "wavs")

In [4]:
df = pd.read_csv(meta_path, sep='|', header=None, names=['text', 'file_id', 'text_norm'])
df['file_id']    = df['file_id'].str.strip().str.replace('wavs/', '', regex=False)
df['audio_path'] = df['file_id'].apply(lambda x: os.path.join(wavs_dir, x))
df               = df[df['audio_path'].apply(os.path.exists)].reset_index(drop=True)
df['sentence']   = df['text'].str.strip()
print(f"Valid samples: {len(df)}")

Valid samples: 78720


In [15]:
N_SAMPLES = 1000  # set to None to use all 78k
if N_SAMPLES:
    df = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")

Train: 950 | Eval: 50


In [16]:
def df_to_hf(df):
    return Dataset.from_dict({
        'audio':    df['audio_path'].tolist(),
        'sentence': df['sentence'].tolist()
    }).cast_column('audio', Audio(sampling_rate=16000))

dataset = DatasetDict({
    'train': df_to_hf(train_df),
    'eval':  df_to_hf(eval_df)
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 950
    })
    eval: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 50
    })
})


In [17]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Arabic", task="transcribe")
model     = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="Arabic", task="transcribe")
model.config.suppress_tokens    = []

print("Whisper loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Whisper loaded: openai/whisper-small


In [26]:
import torch

# ── Pre-process and cache everything once ─────────────────────────────────────
def preprocess(batch):
    audio = batch['audio']
    batch['input_features'] = processor.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

print("Pre-processing dataset (one time only)...")
dataset_processed = DatasetDict({
    'train': dataset['train'].map(preprocess, remove_columns=['audio', 'sentence']),
    'eval':  dataset['eval'].map(preprocess,  remove_columns=['audio', 'sentence'])
})

# Save to disk so you never redo this
dataset_processed.save_to_disk("/tmp/arabic_asr_processed")
print("Done! Saved to /tmp/arabic_asr_processed")

Pre-processing dataset (one time only)...


Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/950 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

Done! Saved to /tmp/arabic_asr_processed


In [27]:
from datasets import load_from_disk

dataset_processed = load_from_disk("/tmp/arabic_asr_processed")
print(dataset_processed)

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 950
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 50
    })
})


In [28]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


In [29]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 padding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}

print("WER metric ready.")

WER metric ready.


In [35]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir               = "./whisper-small-arabic",
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 8,
    gradient_accumulation_steps = 1,
    learning_rate            = 5e-4,
    warmup_steps             = 100,
    max_steps                = 500,          # increase for full training
    gradient_checkpointing   = False,
    fp16                     = True,          # requires T4/A100 GPU
    eval_strategy      = "steps",
    eval_steps               = 250,
    save_strategy            = "steps",
    save_steps               = 250,
    logging_steps            = 50,
    load_best_model_at_end   = True,
    metric_for_best_model    = "wer",
    greater_is_better        = True,        
    predict_with_generate    = True,
    generation_max_length    = 225,
    report_to                = ["none"],      # change to 'tensorboard' or 'wandb' if needed
    push_to_hub              = False,
    remove_unused_columns    = False,   # ← add this

)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = training_args,
    train_dataset = dataset_processed['train'],
    eval_dataset  = dataset_processed['eval'],
    processing_class     = processor.feature_extractor,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

print("Trainer configured. Ready to train!")

Trainer configured. Ready to train!


In [36]:
trainer.train()

# Save final model & processor
trainer.save_model("./whisper-small-arabic-final")
processor.save_pretrained("./whisper-small-arabic-final")
print("Model saved to ./whisper-small-arabic-final")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
results = trainer.evaluate()
print("\n=== Evaluation Results ===")
for k, v in results.items():
    print(f"  {k}: {v}")